# PyPSA tutorial: Refugee Camp Linear Optimal Power Flow (LOPF)

## Temporal Coupling
Previously, the power flow analysis we have undertaken has been for a single time step (usually the time of peak load/generation). However, in reality, the operation of energy systems is temporally coupled. For example, the action taken by a battery at one timestep affects the state of charge of a battery at the next time step. Likewise, if demand customers are asked to be flexible, their consumption patterns in one time step will likely affect their consumption in another time step (you may have a new peak). Similarly, the operation of a generator may depend on its operation in previous time steps (e.g., due to ramping constraints). To capture these temporal dependencies, we need to perform a multi-period optimization that considers multiple time steps simultaneously. This is where PyPSA's Linear Optimal Power Flow (LOPF) capabilities come into play.

We usually want to minimise the total cost of the system over a period of time. To do this, we will need temporal profiles for both the load and the renewable generation. Import them below.

In [ ]:
import pandas as pd
# import example load data
load_profiles = pd.read_csv("../Data/Demand/bidibidi_load_profiles.csv", index_col=0, parse_dates=True)

# TODO: download PV generation data from renewables.ninja for Bidibidi for 1 year. Use an installed capacity of 1 kW so your output is in kW/kWp (i.e., the capacity factor).
bidibidi_pv = # TODO: read the PV generation data into a pandas DataFrame. Reset the index to that of the load_profiles DataFrame if different.


Let's take a look at the load profiles.

In [ ]:
combined_p_load = load_profiles.filter(like='P').sum(axis=1) / 6
ax = load_profiles.loc[pd.Timestamp('2019-01-01'):pd.Timestamp('2019-01-02')].plot(alpha=0.2)
combined_p_load.loc[pd.Timestamp('2019-01-01'):pd.Timestamp('2019-01-02')].plot(ax=ax, color='black', linewidth=2, label='ADMD')
ax2 = ax.twinx()
bidibidi_pv.loc[pd.Timestamp('2019-01-01'):pd.Timestamp('2019-01-02')].plot(ax=ax2, color='gold')
# ax legend off
ax.legend().set_visible(False)
ax2.legend().set_visible(False)

In [ ]:
# The maximum load is the same as what we calculated in the previous notebook.
print(f"Time of maximum load: {combined_p_load.idxmax()}")
print(f"Maximum load: {combined_p_load.max():.2f} kW")

## Linear Optimal Power Flow (LOPF)
PyPSA can do a lot more than just power flow calculations. It can also optimize the operation (dispatch) and investment of energy systems over multiple time steps. This is done by formulating a linear programming problem that minimizes the total cost of the system while satisfying various constraints. The core theory of optimization is beyond the scope of this course. PyPSA provides documentation https://docs.pypsa.org/latest/user-guide/optimization/overview/. Importantly, to make the optimization problem tractable, PyPSA uses a linearized version of the power flow equations (recall the DC power flow approximation where line losses are neglected and voltage angles are small). This means that the optimization results may not be physically feasible, and a power flow calculation should be performed afterwards to check the feasibility of the solution.

Below we will set up a simple optimisation for the refugee camp that optimises the sizing and operation of a solar PV system and battery storage.

In [ ]:
import pypsa
# create a new network
net = pypsa.Network()

In [ ]:
# we now need to tell PyPSA that we want to do a time series simulation, in PyPSA this is done by setting the snapshot attribute of the network
net.set_snapshots(load_profiles.index)

In [ ]:
# add a carrier for AC
net.add(
    "Carrier",
    "AC",
)

In [ ]:
# create the buses as before
# TODO : add the buses to the network using n.add. Use v_nom=120/1000 to convert 120 V to kV.

# add the lines - this time we will make them extendable by setting s_nom_extendable=True. 
# This means that the optimization will be able to choose the size of the line (and hence the cost) as part of the optimization.
for i in range(6):
    net.add("Line", f"Line {i+1}-{i+2}", bus0=f"Bus {i+1}", bus1=f"Bus {i+2}",
            x=0.37, r=0.37, s_nom_extendable=True, overwrite=True, length=0.05,
            capital_cost= # TODO: include a suitable estimate for the annualised cost of the line (in $/MVA) (factoring in length).
            )


In [ ]:
# add loads - now we pass the timeseries p and q to the p_set and q_set parameters.
for i in range(1, 7):
    net.add("Load", f"Load {i}", bus=f"Bus {i+1}", 
           p_set= # TODO
           q_set= # TODO
           overwrite=True)

In [ ]:
# now we add a generator that is expandable. 
# We need to provide a pu time series output (e.g. a capacity factor) and a capital cost for the optimisation cost function to minimise.
# TODO: create a generator for PV at Bus 1, using the capacity factor time series to set p_max_pu. 
# You should also provide an anualised capital cost for PV and set p_nom_extendable=True so that the optimisation can choose the size of the PV system.


In [ ]:
# add extendable battery storage at bus 1, defined by power capacity and max hours at rated power.
# note, should you wish to optimise both power and energy independently, you need to combine a link and store component.
# here we will make use of the StorageUnit component to just optimise the power capacity and set the energy capacity to be 4 hours at rated power.
net.add("StorageUnit", "Battery", bus="Bus 1",
       p_nom_extendable=True,
       capital_cost=400000,  # annualised capital cost in $/MW
       max_hours=4,  # hours of storage at nominal power
       cyclic_state_of_charge=True, # this ensures conservation of energy, meaning the state of charge at the end of the simulation is the same as at the start (no free energy).
       overwrite=True, 
       )

In [ ]:
# run the dispatch and capacity expansion optimisation (this combines the linear optimal power flow and investment optimisation)
net.optimize()

## Results

In [ ]:
print(f"Optimal PV capacity: {net.generators.p_nom_opt.sum()*1000:.2f} kW")
print(f"Optimal Battery capacity: {net.storage_units.p_nom_opt.sum()*1000:.2f} kW")
print(f"Optimal Battery energy capacity: {net.storage_units.p_nom_opt.sum()*1000*net.storage_units.max_hours.sum():.2f} kWh")
print(f"Optimal Line capacities (kVA): {net.lines.s_nom_opt}")

### Plot generation, battery dispatch and state of charge over time

In [ ]:
import matplotlib.pyplot as plt

fig_start = pd.Timestamp(2019, 5, 20)
fig_end   = pd.Timestamp(2019, 5, 21)

fig, (ax, ax2) = plt.subplots(nrows=2, figsize=(10, 10), gridspec_kw={"height_ratios": [4, 1]})

# 1) Load line on ax
load_series = net.loads_t.p.sum(axis=1).loc[fig_start:fig_end]
load_series.plot(ax=ax, label="Load", color="black")

# 2) Generation + storage as stacked area on the same ax
stack = pd.concat([
    net.generators_t.p.loc[fig_start:fig_end, "Solar"],
    net.storage_units_t.p_dispatch.loc[fig_start:fig_end, "Battery"],
    -net.storage_units_t.p_store.loc[fig_start:fig_end, "Battery"],
], axis=1)

stack.columns = ["Solar", "Battery Discharge", "Battery Charge"]  # labels for legend
stack.plot.area(
    ax=ax, stacked=True, linewidth=0.0,
    color=["gold", "darkorange", "blue"]
)

ax.set_ylabel("Power (MW)")
ax.legend(loc="upper right")

# 3) State of charge on ax2
soc = net.storage_units_t.state_of_charge.loc[fig_start:fig_end, "Battery"]
soc.plot(ax=ax2, color="blue", label="Battery SoC")
ax2.set_ylabel("State of Charge (MWh)")
ax2.set_xlabel("Date")
ax2.legend()

plt.tight_layout()
plt.show()

### Print the voltages at each bus over time
What do you observe about the voltages? Why do you think this is the case?

In [ ]:
ax = net.buses_t['v_mag_pu'].loc[pd.Timestamp(2019, 5, 20):pd.Timestamp(2019, 5, 21), :].plot()
ax.set_ylabel("Voltage (p.u.)")

In [ ]:
# inspect them further at peak load time
net.buses_t.v_mag_pu.loc[combined_p_load.idxmax()]

## 2-step optimization and power flow validation

A common workflow is to first optimise the network with the linearised approximations of n.optimize() and then to run the non-linear power flow calculation on the optimised network with n.pf() for validation, using the power imbalances from the optimisation as setpoints. There is a function that does this for you, n.optimize.optimize_and_run_non_linear_powerflow().

In [ ]:
# TODO: use the optimize_and_run_non_linear_powerflow() function from the pypsa optimize module to validate the results of the linear optimal power flow.


### Replot the results to check that the results are techincally valid.


In [ ]:
ax = net.buses_t['v_mag_pu'].loc[pd.Timestamp(2019, 5, 20):pd.Timestamp(2019, 5, 21), :].plot()
ax.set_ylabel("Voltage (p.u.)")

In [ ]:
# inspect them further at peak load time
net.buses_t.v_mag_pu.loc[combined_p_load.idxmax()]

## Extension
Rather than the battery being at the hub. Try adding battery storage at a different node (i.e. in one of the homes).  

Rerun the optimisation and see if the system changes. How do you make sense of the results.

In [ ]:
net_ge = pypsa.Network()
net_ge.set_snapshots(load_profiles.index)
net_ge.add(
    "Carrier",
    "AC",
)
# create the buses as before
for i in range(7):
    net_ge.add("Bus", f"Bus {i+1}", v_nom=120/1000, x=i, y=0,)

# add the lines - this time we will make them extendable by setting s_nom_extendable=True. 
for i in range(6):
    net_ge.add("Line", f"Line {i+1}-{i+2}", bus0=f"Bus {i+1}", bus1=f"Bus {i+2}",
            x=0.37, r=0.37, s_nom_extendable=True, overwrite=True, length=0.05,
            capital_cost=2000 * 0.05, # $/MVA based on $2000 $ per MVA-km per year
            )
# add loads - now we set the p to be the time series power profiles
for i in range(1, 7):
    net_ge.add("Load", f"Load {i}", bus=f"Bus {i+1}", 
           p_set=load_profiles[f"Load {i} P"].values/1000**2,
           q_set=load_profiles[f"Load {i} Q"].values/1000**2,
           overwrite=True)

net_ge.add("Generator", "Solar", bus="Bus 1", 
       p_max_pu=bidibidi_pv["electricity"].values,
       p_nom_extendable=True,
       capital_cost=110000, # annualised capital cost in $/MW
       ) 


In [ ]:
# TODO: add a battery storage unit at a different bus and rerun the optimisation. How does the system change?

In [ ]:
net_ge

In [ ]:
# TODO: rerun the optimisation and validation function

In [ ]:
# Check the voltage at peak demand 
net_ge.buses_t.v_mag_pu.loc[combined_p_load.idxmax()]

In [ ]:
ax = net_ge.buses_t['v_mag_pu'].loc[pd.Timestamp(2019, 5, 20):pd.Timestamp(2019, 5, 21), :].plot()
ax.set_ylabel("Voltage (p.u.)")